# Pipeline and Models

Run the full Phase 1 + 2 pipeline from **VS Code (Jupyter extension)**, terminal, or Colab.

## What "10,000" means in this project

We are **not** training on all 33,935 influencers or ~10M mapping rows. The modeling subset is:

- **~10,000 Instagram posts** (post-level records with captions, likes, comments, etc.)

That is what your original Colab notebook sampled from `posts_info.zip`. The pipeline supports that size; local synthetic mode uses fewer posts unless you increase `SYNTHETIC_INFLUENCERS` below.

## Modes

| Mode | When | Approx. posts |
|------|------|----------------|
| `real_metadata` | Colab + extracted `.info` files | up to `TARGET_POSTS` (default 20,000) |
| `parquet` | Reuse `posts_base_*.parquet` | whatever was saved |
| `synthetic` | Local run (default) | exactly `TARGET_POSTS` (default 20,000) |

Legacy notebooks (`Data_Extraction.ipynb`, `Cleaned Up Notebook.ipynb`) are Colab-only and were not updated.

In [ ]:
from pathlib import Path
import sys

# --- Configuration ---
# Pick one data mode: "synthetic" | "parquet" | "real_metadata"
DATA_MODE = "synthetic"

# For synthetic local runs (~7k posts with 500 influencers; ~1.7k with 120)
SYNTHETIC_INFLUENCERS = 120

# Target post count for synthetic generation or real-metadata parsing cap
TARGET_POSTS = 20_000

# For real 10k Colab runs, set DATA_MODE = "real_metadata" and this path:
EXTRACTED_METADATA_DIR = None  # e.g. Path("/content/Post_metadata_10000_extracted")

# Or reuse a saved parquet from a prior Colab run:
POSTS_PARQUET = None  # e.g. OUTPUT_DIR / "processed" / "posts_base_10000.parquet"

TOP_K = 5
RANDOM_SEED = 42


def find_repo_root() -> Path:
    """Works from VS Code, terminal jupyter, or Colab regardless of cwd."""
    candidates: list[Path] = [Path.cwd(), *Path.cwd().parents]
    try:
        candidates.insert(0, Path(__file__).resolve().parent.parent)
    except NameError:
        pass

    colab_repo = Path("/content/drive/MyDrive/DSCI351/Class Project/DSCI_Recommender_System_Class_Proj")
    if colab_repo.exists():
        candidates.insert(0, colab_repo)

    for candidate in candidates:
        if (candidate / "src" / "pipeline.py").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find repo root. Open this notebook from the project repo or set cwd to notebooks/."
    )


REPO = find_repo_root()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

DATA_DIR = REPO / "data"
OUTPUT_DIR = REPO / "artifacts"

if DATA_MODE == "parquet" and POSTS_PARQUET is None:
    processed_dir = OUTPUT_DIR / "processed"
    if processed_dir.exists():
        existing = sorted(processed_dir.glob("posts_base_*.parquet"))
        if existing:
            POSTS_PARQUET = existing[-1]

USE_SYNTHETIC = DATA_MODE == "synthetic"
USE_PARQUET = DATA_MODE == "parquet" and POSTS_PARQUET is not None
USE_EXTRACTED = DATA_MODE == "real_metadata" and EXTRACTED_METADATA_DIR is not None

if DATA_MODE == "real_metadata" and not USE_EXTRACTED:
    raise ValueError("Set EXTRACTED_METADATA_DIR for real_metadata mode.")
if DATA_MODE == "parquet" and not USE_PARQUET:
    raise ValueError("Set POSTS_PARQUET (or save a posts_base_*.parquet) for parquet mode.")

print("Repo:", REPO)
print("Data dir exists:", DATA_DIR.exists())
print("Influencers file exists:", (DATA_DIR / "influencers.txt").exists())
print("DATA_MODE:", DATA_MODE)
if USE_PARQUET:
    print("POSTS_PARQUET:", POSTS_PARQUET)
if USE_EXTRACTED:
    print("EXTRACTED_METADATA_DIR:", EXTRACTED_METADATA_DIR)
if USE_SYNTHETIC:
    print("SYNTHETIC_INFLUENCERS:", SYNTHETIC_INFLUENCERS)

In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-r",
    str(REPO / "requirements.txt"),
])
print("Dependencies installed from", REPO / "requirements.txt")

In [ ]:
from src.pipeline import PipelineConfig, run_pipeline

config = PipelineConfig(
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    target_posts=TARGET_POSTS,
    posts_parquet=POSTS_PARQUET if USE_PARQUET else None,
    extracted_metadata_dir=EXTRACTED_METADATA_DIR if USE_EXTRACTED else None,
    synthetic=USE_SYNTHETIC,
    synthetic_influencers=SYNTHETIC_INFLUENCERS,
    k=TOP_K,
    seed=RANDOM_SEED,
)

outputs = run_pipeline(config)

posts_base_df = outputs["posts_base_df"]
print(f"Posts in dataset: {len(posts_base_df):,}")
print(f"Unique influencers: {posts_base_df['influencer_name'].nunique():,}")
print(f"Unique strategies: {posts_base_df['strategy'].nunique():,}")
if USE_SYNTHETIC:
    print("Note: local synthetic run — for ~10,000 real posts, use Colab + DATA_MODE='real_metadata'.")
elif len(posts_base_df) < 9000:
    print("Note: dataset is below ~10k posts; OK for testing, but use full Colab extract for final report.")

outputs["results_df"]

In [ ]:
try:
    from IPython.display import Image, display

    print("Hybrid alpha:", outputs["hybrid_alpha"])
    print("Results saved to:", outputs["results_path"])
    for figure_path in outputs["figure_paths"]:
        display(Image(filename=str(figure_path)))
except ImportError:
    print("Hybrid alpha:", outputs["hybrid_alpha"])
    print("Results saved to:", outputs["results_path"])
    for figure_path in outputs["figure_paths"]:
        print("Figure:", figure_path)